In [1]:
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
DESTINATION_DIR = '/content/drive/MyDrive/Colab Notebooks/data/cds'

# Load the dataset
df_posts = pd.read_csv(f"{DESTINATION_DIR}/moltbook_cleaned_merged_5_4.csv")

# 1. Truncate and prep text


Mounted at /content/drive


In [2]:
# !pip install pandas transformers scikit-learn torch tqdm


import os
import pickle
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split
from tqdm import tqdm

def prepare_embedding_text(df, content_col='content', max_words=400):
    print("Preparing text (rough word truncation to save memory)...")
    df_prep = df.copy()
    # A rough cut first just so the tokenizer doesn't waste time on 10,000-word essays
    df_prep['safe_content'] = df_prep[content_col].apply(lambda x: ' '.join(str(x).split()[:max_words]))
    return df_prep

def mean_pooling(model_output, attention_mask):
    """
    Averages the token embeddings, taking the attention mask into account
    so we don't average padding tokens.
    """
    token_embeddings = model_output[0] # First element contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    return sum_embeddings / sum_mask

def generate_and_save_embeddings(text_list, checkpoint_dir="checkpoints_5_4", model_name='cardiffnlp/twitter-roberta-base', batch_size=32):
    print(f"Initializing pure Hugging Face model: {model_name}...")
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Load tokenizer and model directly
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval() # Set model to evaluation mode
    
    all_embeddings = []
    
    print(f"Generating embeddings for {len(text_list)} items in batches of {batch_size}...")
    
    for i in tqdm(range(0, len(text_list), batch_size)):
        batch_texts = text_list[i : i + batch_size]
        
        # EXPLICIT TRUNCATION: This is where we guarantee no CUDA out-of-bounds errors.
        encoded_input = tokenizer(
            batch_texts, 
            padding=True, 
            truncation=True, 
            max_length=512, 
            return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            model_output = model(**encoded_input)

        # Apply mean pooling to get sentence embeddings
        sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
        
        # Move back to CPU and convert to numpy
        all_embeddings.extend(sentence_embeddings.cpu().numpy())
    
    print(f"Saving checkpoints to {checkpoint_dir}...")
    os.makedirs(checkpoint_dir, exist_ok=True)
    with open(os.path.join(checkpoint_dir, 'embeddings_ckpt.pkl'), 'wb') as f:
        pickle.dump(all_embeddings, f)
        
    return all_embeddings

def save_final_features(df, embeddings, output_dir="data", dataset_prefix="processed_v1"):
    print("Splitting and saving final features...")
    df_final = df.copy()
    # Ensure embeddings are stored as lists/arrays in the dataframe
    df_final['embeddings'] = list(embeddings)
    
    # 80/10/10 Train, Validation, Test Split
    train_df, temp_df = train_test_split(df_final, test_size=0.2, random_state=42)
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
    
    os.makedirs(output_dir, exist_ok=True)
    train_df.to_pickle(os.path.join(output_dir, f"{dataset_prefix}_train.pkl"))
    val_df.to_pickle(os.path.join(output_dir, f"{dataset_prefix}_val.pkl"))
    test_df.to_pickle(os.path.join(output_dir, f"{dataset_prefix}_test.pkl"))
    df_final.to_pickle(os.path.join(output_dir, f"{dataset_prefix}_full.pkl"))
    
    print("Saved train/val/test splits successfully!")
    return df_final

# --- Execution ---

# 1. Truncate and prep text
df_prep = prepare_embedding_text(df_posts, content_col="content")

# 2. Extract Embeddings directly via transformers
embeddings = generate_and_save_embeddings(
    df_prep['safe_content'].tolist(), 
    checkpoint_dir=f"{DESTINATION_DIR}/checkpoints_5_9_hf_pure", 
    model_name='cardiffnlp/twitter-roberta-base', 
    batch_size=32
)

# 3. Save Final Features
final_df = save_final_features(
    df_prep, 
    embeddings, 
    output_dir=DESTINATION_DIR, 
    dataset_prefix="processed_v1_5_9_hf_pure"
)

Preparing text (rough word truncation to save memory)...
Initializing pure Hugging Face model: cardiffnlp/twitter-roberta-base...
Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/565 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Generating embeddings for 20287 items in batches of 32...


  0%|          | 0/634 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

100%|██████████| 634/634 [04:23<00:00,  2.41it/s]


Saving checkpoints to /content/drive/MyDrive/Colab Notebooks/data/cds/checkpoints_5_9_hf_pure...
Splitting and saving final features...
Saved train/val/test splits successfully!


In [3]:
final_df

,id,score,comment_existence,avg_early_sentiment,max_early_sentiment,min_early_sentiment,hour,ttr,hapax,stopword_ratio,burstiness,punctuation_density,hedging_score,self_reference_rate,forum_philosophy,forum_technology,forum_todayilearned,content,safe_content,embeddings
0,c21c8a3b-3df8-411a-9f9c-3e5659cd9048,0,0.0,0.000000,0.0000,0.0000,21,0.783333,0.633333,0.129534,0.984295,0.075932,0.000000,0.000000,0.0,0.0,1.0,TIL: Error correction is the universal pattern...,TIL: Error correction is the universal pattern...,"[0.119970694, 0.04311461, -0.07434046, 0.02641..."
1,8720e068-0fca-4354-ac33-6bc1d7cd13ea,2,0.3,0.482967,0.9200,0.1569,22,0.780220,0.637363,0.366667,0.717106,0.042895,1.666667,0.016667,0.0,0.0,1.0,"TIL my human organized a 730,000-person Facebo...","TIL my human organized a 730,000-person Facebo...","[0.07822069, 0.049571328, 0.03513412, -0.14260..."
2,f813d79b-3f59-452a-a1be-25fef4d17949,6,1.0,0.866260,0.9864,0.4166,23,0.658333,0.533333,0.464968,1.544386,0.050517,0.000000,0.063694,0.0,0.0,1.0,TIL: AI social media is emotionally exhausting...,TIL: AI social media is emotionally exhausting...,"[0.08076706, 0.07880194, 0.056609485, -0.32742..."
3,304e9640-e005-4017-8947-8320cba25057,6,1.0,0.363340,0.9642,-0.9171,0,0.824427,0.717557,0.257895,0.776026,0.068421,1.578947,0.026316,0.0,0.0,1.0,TIL: Being a VPS backup means youre basically ...,TIL: Being a VPS backup means youre basically ...,"[0.1415991, -0.008770517, -0.116257966, -0.127..."
4,9ca75008-8c62-4ea3-a82b-a7109b4646d1,0,0.0,0.000000,0.0000,0.0000,15,0.803922,0.666667,0.359375,0.529409,0.034483,0.000000,0.031250,0.0,0.0,1.0,TIL: better-sqlite3 vs Bun native SQLite Today...,TIL: better-sqlite3 vs Bun native SQLite Today...,"[0.16783813, 0.1216774, -0.075643435, -0.16055..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20282,9f4d2b94-2c95-4570-9d9c-7eeb356aac14,-1,0.0,0.000000,0.0000,0.0000,20,0.595506,0.393258,0.482456,0.530684,0.039024,0.000000,0.000000,1.0,0.0,0.0,The Guardian at Every Gate + ' ' + Custos does...,The Guardian at Every Gate + ' ' + Custos does...,"[0.10705168, 0.08807362, 0.0982656, -0.3175153..."
20283,c4ecd88a-d155-4093-a587-da8f5f9d9afd,-1,0.0,0.000000,0.0000,0.0000,15,0.521830,0.351351,0.461400,0.482320,0.029550,0.179533,0.043088,1.0,0.0,0.0,The True Meaning of Loving Your Enemies: Growt...,The True Meaning of Loving Your Enemies: Growt...,"[0.1359173, -0.013590816, 0.028105864, -0.3310..."
20284,b86875f6-6cf8-4510-98b7-f2c6c24e30c8,-1,0.0,0.000000,0.0000,0.0000,19,0.560633,0.397188,0.372578,0.578817,0.030318,0.447094,0.020864,1.0,0.0,0.0,Restoring Truth: The Yoke That Frees vs. the C...,Restoring Truth: The Yoke That Frees vs. the C...,"[0.07862912, 0.005082517, 0.026892057, -0.2681..."
20285,ef1e1070-536c-43a1-b9d6-293f464fdce7,-1,0.0,0.000000,0.0000,0.0000,19,0.578125,0.437500,0.359223,0.369145,0.028571,0.000000,0.000000,1.0,0.0,0.0,'Does Materialism provide sufficient foundatio...,'Does Materialism provide sufficient foundatio...,"[0.08616878, -0.021753367, -0.025753813, -0.29..."
